# Overhead plots

The idea is to create a plot for the CPU and latency overhead when there's an Agent actually taking decisions and when there is no Agent.

The first approximation is: when there is no agent --> when the agent picks always the same value, in that case I am assuming the cost is basically 0. I am saying this because we measure the CPU consumption of the Aggregate, and if the D value does not change there is not really much the Aggregate is doing.

Of course it would be more "correct" to do experiments with an Aggregate that is not connected at all with the RL framework part, but in the interest of time we start with this.

Now, we need to go back to the actual log data and see a bit what we have... the data is here:  `data/10/linearroad-CCR/5/600`. From the data:
- we have all the D values
- we have 100 episodes for the D value
- even when D is 10, the episode runs for some 80 seconds

Hence:
- we could take the middle 60 seconds from each D and from each episode and log them
- we already have the python script that "cuts" individual episodes from the single log file, so we could add an optional parameter that asks if we want to dump the individual logs (default: no)
- then we do the same for the RL agent experiment and we have what we need to start creating the new plots

To avoid messing with the data we already have, I am creating a copy of the folder so we create the new data in the new folder. I'm copying it into `data/exp016.22.overhead/linearroad/baseline`

- `mkdir data/exp016.22.overhead`
- `mkdir data/exp016.22.overhead/linearroad`
- `mkdir data/exp016.22.overhead/linearroad/baseline`
- `cp -r data/10/linearroad-CCR/5/600/* data/exp016.22.overhead/linearroad/baseline/`

Then I am running:
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/linearroad/baseline True 0` but changing `reward_pattern` to `"Old` to see what happens

This seems to work
Now I'm adding --dumpdata to `plot_experiment_stats_exp.py` to dump data too (if the paramater is passed)

This is how you run it:
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/linearroad/baseline True 0 1 2 3 4 5 6 7 8 9 10`

Now doing the same for synthetic

And now trying with the agent one, again:
- copy the data
- Run the script (this time with the New not the Old parameter)
- `mkdir data/exp016.22.overhead/linearroad/agent`
- `cp -r data/exp16-5/WELAW/linear/* data/exp016.22.overhead/linearroad/agent/`
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/linearroad True agent`
- `mkdir data/exp016.22.overhead/synthetic/agent`
- `cp -r data/exp16-5/WELAW/synthetic/* data/exp016.22.overhead/synthetic/agent/`
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/synthetic True agent`



# 250107

- I just realized the way I wanted to prepare the plots is actually not the best, because we might be comparing executions at completely different points in times, in which CPU and Latency are different not because of the Agent, but because of the different point in time in the exectuion
- We need to run from scratch an Aggregate without nothing and one with the Agent
- Let's start running one without nothing, what we could do is that
  - can we define a single episode and a single step but make the inter-step period long enough?
  - can we force the random seed to be a specific value to make sure the experiment is always starting at the same point?
  - We should also think about the policy barrier, if that plays a role

So now:
- I can run a single episode of the length I want with basically no action taking place
- I can choose the random seed and the state measurement check period
- The next would be
  - have a random that takes random steps but only increase/decrease/stay --> Actually this is already what is happenning it seems, will doublecheck but it should be like that
  - setup a new experiment in which there is still 1 episode but a certain number of steps
  - and here I would use the WELAW policy by the way
  - then we should have the data for one repetition (NOTE: one episode per run, since it will be difficult to align episodes across runs if one does only one step and the other an arbitrary number of steps)
  - note also that if X is the state monitoring check period the experiment will be 2X long with an action at X, so we should take the data during the first X I think

Now I can:
- reuse the script from yesterday to extract the data (putting Old in the internal parameter of create_plots_for_exp)
- `./scripts/create_plots_for_exp.sh data/overhead/linearroad/5/600 True 10 r`

- From the results it seems to work, but the experiment length for the "no-agent case" is strange...
- It might be because of the time it takes in between the reset and the time the SPE is actually ready... Maybe 2 actions just to double check

Now I should run a couple of experiments, before starting creating plots, so the same setup but for a bunch of random seeds I guess
- running start_all_evaluation_CCR.sh
- `./scripts/create_plots_for_exp.sh data/overhead/linearroad True 10/2 10/5 10/122 10/123 10/1242 r/2 r/5 r/122 r/123 r/1242`

Now creating a python script to merge the data

Write a python script that using argsparse gets:
- a folder A, a folder B, a series of experiments ids, an output csv file data
- for each id, reads the files A/id/eps/CPU-agg.average.000.csv and B/id/eps/CPU-agg.average.000.csv (there are 2 columns, timestamp and value, with headers) and puts the column i and the column timestamp in the output csv file and the value column of each file as extra columns in the output csv. Note in the file name i uses 3 digits and also note that the timestamp column should contain the union of the timestamp columns found in the two file
- then it also adds the columns taken from the files A/id/eps/latency.average.000.csv and B/id/eps/latency.average.000.csv

- `python plotting/merge_overhead_data.py data/overhead/linearroad/10 data/overhead/linearroad/r 2 5 122 123 1242 data/overhead/linearroad/merged.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead/linearroad/merged.csv data/overhead/linearroad/diffs.csv`

Now doing the same with synthetic:
- `./scripts/start_all_evaluation_CCR.sh` (after updating the internal parameters to synthetic)
- `./scripts/create_plots_for_exp.sh data/overhead/synthetic True 10/2 10/5 10/122 10/123 10/1242 r/2 r/5 r/122 r/123 r/1242`
- `python plotting/merge_overhead_data.py data/overhead/synthetic/10 data/overhead/synthetic/r 2 5 122 123 1242 data/overhead/synthetic/merged.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead/synthetic/merged.csv data/overhead/synthetic/diffs.csv`

# 250107

- one question: in the process I am using all the data from each episode or only the central portion?
  - Yes, and I can see two problems as of now
    - For the synthetic use case I am not running enough steps
    - I should only take the periods in which the latency is smaller than the hard threshold and the CPU is less than 0.99 or whatever, because the episdoe would terminate there

# ALL THE REST IS OLD

## What rewards do we get? Are these the expected ones?

- `python plotting/actions_rewards.py data/exp16.20_shepherding`

- From SPE always 0
- From Agent high variability, as expeted. Cannot say if negative but that can be checked later on with the following graphs

## Are the durations across actions the expected ones?

- `python plotting/actions_times.py data/exp16.20_shepherding`

Usual ones, yes

## Termination
- How many terminate because of CPU?
- How many terminate because of latency?
- How many terminate because of CPU & Latency?

Copy paste the following in temp.sh and run as temp.sh data/exp16.20_shepherding

```
find $1 -type f -name "python_agent.log" -exec bash -c '
  for file; do
    echo "Processing $file"
    cpu_count=$(grep -c "High cpu observed" "$file")
    latency_count=$(grep -c "High latency observed" "$file")
    echo "High CPU observed: $cpu_count"
    echo "High Latency observed: $latency_count"
  done
' bash {} +
```


```
Processing data/exp16.20//WELOB/linear/python_agent.log
High CPU observed: 73
High Latency observed: 1
Processing data/exp16.20//ELOB/linear/python_agent.log
High CPU observed: 108
High Latency observed: 21
Processing data/exp16.20//LOB/linear/python_agent.log
High CPU observed: 115
High Latency observed: 29
Processing data/exp16.20//WELAW/linear/python_agent.log
High CPU observed: 120
High Latency observed: 4
```
It's mostly CPU, and the longer the inter-action time, the higher the number of early termination because of CPU. The threshold is 90.0

### Prepare data

Copy paste the following in temp.sh and run 

```
./scripts/create_plots_for_exp.sh data/exp16.20_shepherding False WELOB/linear
python plotting/create_summary_data.py data/exp16.20_shepherding/WELOB
./scripts/create_plots_for_exp.sh data/exp16.20_shepherding False ELOB/linear
python plotting/create_summary_data.py data/exp16.20_shepherding/ELOB
./scripts/create_plots_for_exp.sh data/exp16.20_shepherding False LOB/linear
python plotting/create_summary_data.py data/exp16.20_shepherding/LOB
./scripts/create_plots_for_exp.sh data/exp16.20_shepherding False WELAW/linear
python plotting/create_summary_data.py data/exp16.20_shepherding/WELAW
sed -i '' 's/linear/welob_linear/g' data/exp16.20_shepherding/WELOB/baselines_data.csv
sed -i '' 's/linear/elob_linear/g' data/exp16.20_shepherding/ELOB/baselines_data.csv
sed -i '' 's/linear/lob_linear/g' data/exp16.20_shepherding/LOB/baselines_data.csv
sed -i '' 's/linear/welaw_linear/g' data/exp16.20_shepherding/WELAW/baselines_data.csv
find data/exp16.20_shepherding/ -type f -name "baselines_data.csv" -exec cat {} + > data/exp16.20_shepherding/merged.csv
python plotting/plot_probs_evolution.py data/exp16.20_shepherding
python plotting/plot_cumulative_reward.py data/exp16.20_shepherding/ 1001 3100
```

## Steps and rewards

In [1]:
from IPython.display import display, Markdown

# Define your image folder path
path_to_image = "../data/exp16.20_shepherding/"

# Create a Markdown string with the table and images
markdown_table = f"""
<table>
    <tr>
        <td><img src="{path_to_image}WELOB_linear.steps.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}ELOB_linear.steps.png" alt="Image 2" width="450"/></td>
        <td><img src="{path_to_image}LOB_linear.steps.png" alt="Image 3" width="450"/></td>
        # <td><img src="{path_to_image}WELAW_linear.steps.png" alt="Image 3" width="450"/></td>
    </tr>
    <tr>
        <td><img src="{path_to_image}WELOB_linear.reward.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}ELOB_linear.reward.png" alt="Image 2" width="450"/></td>
        <td><img src="{path_to_image}LOB_linear.reward.png" alt="Image 3" width="450"/></td>
        # <td><img src="{path_to_image}WELAW_linear.reward.png" alt="Image 3" width="450"/></td>
    </tr>
</table>
"""

# Render the Markdown
display(Markdown(markdown_table))



<table>
    <tr>
        <td><img src="../data/exp16.20_shepherding/WELOB_linear.steps.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/ELOB_linear.steps.png" alt="Image 2" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/LOB_linear.steps.png" alt="Image 3" width="450"/></td>
        # <td><img src="../data/exp16.20_shepherding/WELAW_linear.steps.png" alt="Image 3" width="450"/></td>
    </tr>
    <tr>
        <td><img src="../data/exp16.20_shepherding/WELOB_linear.reward.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/ELOB_linear.reward.png" alt="Image 2" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/LOB_linear.reward.png" alt="Image 3" width="450"/></td>
        # <td><img src="../data/exp16.20_shepherding/WELAW_linear.reward.png" alt="Image 3" width="450"/></td>
    </tr>
</table>


Seems consistent with stuff seen before

In [2]:
from IPython.display import display, Markdown

# Define your image folder path
path_to_image = "../data/exp16.20_shepherding/"

# Create a Markdown string with the table and images
markdown_table = f"""
<table>
    <tr>
        <td><img src="{path_to_image}WELOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}WELOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}WELOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="{path_to_image}ELOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}ELOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}ELOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="{path_to_image}LOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}LOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}LOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="{path_to_image}WELAW/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}WELAW/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}WELAW/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
</table>
"""

# Render the Markdown
display(Markdown(markdown_table))



<table>
    <tr>
        <td><img src="../data/exp16.20_shepherding/WELOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/WELOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/WELOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="../data/exp16.20_shepherding/ELOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/ELOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/ELOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="../data/exp16.20_shepherding/LOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/LOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/LOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="../data/exp16.20_shepherding/WELAW/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/WELAW/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/WELAW/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
</table>


## final graph

- modify manually the merged file (remove the extra baseline,... lines)
- `python plotting/paper_plot_baselines_vs_multiple_agents_icpe_single_column.py data/10/linearroad-CCR/5/600 data/10/linearroad-CCR/5/600/lr_rate.csv data/exp16.20_shepherding/merged.csv data/exp16.20_shepherding/lr_baseline_vs_multiple_agents.pdf data/exp16.20_shepherding/lr_baseline_vs_multiple_agents.png linearroad welaw_linear welob_linear,elob_linear,lob_linear,welaw_linear WEL-OB,EL-OB,L-OB,WEL-AW`

In [3]:
from IPython.display import display, Markdown

# Define your image folder path
path_to_image = "../data/exp16.20_shepherding/"

# Create a Markdown string with the table and images
markdown_table = f"""
<table>
    <tr>
        <td><img src="{path_to_image}lr_baseline_vs_multiple_agents.png" alt="Image 1" width="450"/></td>
    </tr>
</table>
"""

# Render the Markdown
display(Markdown(markdown_table))



<table>
    <tr>
        <td><img src="../data/exp16.20_shepherding/lr_baseline_vs_multiple_agents.png" alt="Image 1" width="450"/></td>
    </tr>
</table>


Modify the script so that we mark the areas in which the moving average is below the threshold (will be relevant for linear I think)